In [16]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Read Parquet").getOrCreate()

data = spark.read.parquet("data/yellow_tripdata_2026-05.parquet")
data.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



# Calculating trip duration and revenue per minute

In [17]:
from pyspark.sql import functions as F
data1 = data
data1 = data1.withColumn('duration_minute', F.timestamp_diff('MINUTE', 'tpep_pickup_datetime', 'tpep_dropoff_datetime'))
zone_stats = data1.groupby('PULocationID').agg({'total_amount': 'sum', 'duration_minute': 'sum'}).withColumnsRenamed({'sum(duration_minute)': 'total_min', 'sum(total_amount)': 'total_revenue'}) 
zone_stats = zone_stats.withColumn('revenue_per_min', F.col('total_revenue')/F.col('total_min'))
zone_stats.show()

+------------+------------------+---------+------------------+
|PULocationID|     total_revenue|total_min|   revenue_per_min|
+------------+------------------+---------+------------------+
|         148| 1625090.369999921|   925106|1.7566531510982752|
|         243|182600.03999999957|   121396|1.5041685063758243|
|          31|           4318.01|     3192|1.3527600250626568|
|         137| 1216915.250000026|   762331|1.5963082309390881|
|          85| 52747.40999999998|    55030|0.9585209885516988|
|         251| 610.9599999999999|      375|1.6292266666666664|
|          65| 197822.1299999997|   140600|1.4069852773826437|
|         255|367250.70000000356|   219121|1.6760178166401374|
|          53|          11517.03|     8948|1.2871066160035762|
|         133| 31841.34000000001|    28202| 1.129045457768953|
|          78|38699.979999999974|    46716|0.8284095384878837|
|         155|          38836.72|    46281|0.8391504072945701|
|         108|23932.820000000003|    32810|0.7294367570

In [18]:
import geopandas as gpd
taxi_zones = gpd.read_file('data/taxi_zones/taxi_zones.shp')
print(taxi_zones.dtypes)
taxi_zones.head()

OBJECTID         int32
Shape_Leng     float64
Shape_Area     float64
zone               str
LocationID       int32
borough            str
geometry      geometry
dtype: object


,OBJECTID,Shape_Leng,Shape_Area,zone,LocationID,borough,geometry
0,1,0.116357,0.000782,Newark Airport,1,EWR,"POLYGON ((933100.918 192536.086, 933091.011 19..."
1,2,0.433470,0.004866,Jamaica Bay,2,Queens,"MULTIPOLYGON (((1033269.244 172126.008, 103343..."
2,3,0.084341,0.000314,Allerton/Pelham Gardens,3,Bronx,"POLYGON ((1026308.77 256767.698, 1026495.593 2..."
3,4,0.043567,0.000112,Alphabet City,4,Manhattan,"POLYGON ((992073.467 203714.076, 992068.667 20..."
4,5,0.092146,0.000498,Arden Heights,5,Staten Island,"POLYGON ((935843.31 144283.336, 936046.565 144..."


In [19]:
neighbours = gpd.sjoin(
    taxi_zones[['OBJECTID', 'geometry']],
    taxi_zones[['OBJECTID', 'geometry']],
    how='left',
    predicate='touches'
)
neighbours_id = (
    neighbours
    .groupby('OBJECTID_left')['OBJECTID_right']
    .agg(list)
    .reset_index()
    .rename(columns={'OBJECTID_left': 'ID', 'OBJECTID_right': 'neighbour_ID'})
)
print(neighbours_id.dtypes)
neighbours_id

ID               int32
neighbour_ID    object
dtype: object


,ID,neighbour_ID
0,1,[nan]
1,2,[132.0]
2,3,"[242.0, 184.0, 51.0, 254.0]"
3,4,"[148.0, 79.0, 224.0]"
4,5,[nan]
...,...,...
258,259,[254.0]
259,260,[157.0]
260,261,"[88.0, 209.0, 231.0]"
261,262,"[140.0, 141.0, 263.0, 75.0]"


In [20]:
import math
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    ArrayType
)


# ============================================================
# 1. Convert GeoPandas neighbour data into Python records
# ============================================================

records = []

for _, row in neighbours_id.iterrows():

    neighbour_list = row["neighbour_ID"]

    # Remove NaN values from the neighbour list
    if isinstance(neighbour_list, list):
        cleaned_neighbours = [
            int(x)
            for x in neighbour_list
            if x is not None
            and not (isinstance(x, float) and math.isnan(x))
        ]
    else:
        cleaned_neighbours = []

    records.append(
        (
            int(row["ID"]),
            cleaned_neighbours
        )
    )


# ============================================================
# 2. Create Spark DataFrame containing the neighbours
# ============================================================

neighbour_schema = StructType([
    StructField(
        "ID",
        IntegerType(),
        nullable=False
    ),
    StructField(
        "neighbour_ID",
        ArrayType(IntegerType()),
        nullable=True
    )
])

neighbours_spark = spark.createDataFrame(
    records,
    schema=neighbour_schema
)


# ============================================================
# 3. Turn each neighbour in the list into its own row
# ============================================================

neighbours_exploded = neighbours_spark.select(
    "ID",
    F.explode("neighbour_ID").alias("neighbour_ID")
)


# ============================================================
# 4. Get revenue_per_min for each neighbour
# ============================================================

neighbour_revenue = neighbours_exploded.join(
    zone_stats.select(
        F.col("PULocationID"),
        F.col("revenue_per_min").alias(
            "neighbour_revenue_per_min"
        )
    ),
    neighbours_exploded["neighbour_ID"]
        == F.col("PULocationID"),
    how="left"
)


# ============================================================
# 5. Calculate average revenue_per_min across neighbours
# ============================================================

expected_revenue = neighbour_revenue.groupBy(
    "ID"
).agg(
    F.avg(
        "neighbour_revenue_per_min"
    ).alias(
        "expected_revenue_per_min"
    )
)


# ============================================================
# 6. Add expected revenue back onto zone_stats
# ============================================================

zone_stats = zone_stats.join(
    expected_revenue,
    zone_stats["PULocationID"]
        == expected_revenue["ID"],
    how="left"
).drop(
    expected_revenue["ID"]
)


# ============================================================
# 7. Check the result
# ============================================================

zone_stats.select(
    "PULocationID",
    "revenue_per_min",
    "expected_revenue_per_min"
).show()

+------------+------------------+------------------------+
|PULocationID|   revenue_per_min|expected_revenue_per_min|
+------------+------------------+------------------------+
|         148|1.7566531510982752|       1.653501339299507|
|         243|1.5041685063758243|                    NULL|
|          31|1.3527600250626568|      0.8802542748103541|
|         137|1.5963082309390881|      1.6992549744056626|
|          85|0.9585209885516988|      0.9264839113916488|
|         251|1.6292266666666664|      2.1012967665733027|
|          65|1.4069852773826437|       1.299148583297166|
|         255|1.6760178166401374|      1.5873859718485757|
|          53|1.2871066160035762|      2.0306393371950735|
|         133| 1.129045457768953|      0.8490457330872127|
|          78|0.8284095384878837|      0.9381596432942823|
|         155|0.8391504072945701|      0.7837298173399943|
|         108|0.7294367570862543|      0.7425317826014546|
|         211|1.6490804386044262|      1.698555043084175

In [21]:
zone_stats = zone_stats.withColumn(
    "underperformance_pct",
    ((F.col("expected_revenue_per_min") - F.col("revenue_per_min")) 
     / F.col("expected_revenue_per_min") * 100)
)

zone_stats = zone_stats.withColumn(
    "is_underperformer",
    F.col("underperformance_pct") > 10  # Flag zones >10% below neighbors
)

In [22]:
city_avg = zone_stats.agg(
    F.avg("revenue_per_min").alias("city_avg_revenue")
).collect()[0][0]

zone_stats = zone_stats.withColumn(
    "city_avg_revenue",
    F.lit(city_avg)
).withColumn(
    "below_city_avg_pct",
    ((F.col("city_avg_revenue") - F.col("revenue_per_min")) 
     / F.col("city_avg_revenue") * 100)
)
zone_stats.show()

+------------+------------------+---------+------------------+------------------------+--------------------+-----------------+------------------+--------------------+
|PULocationID|     total_revenue|total_min|   revenue_per_min|expected_revenue_per_min|underperformance_pct|is_underperformer|  city_avg_revenue|  below_city_avg_pct|
+------------+------------------+---------+------------------+------------------------+--------------------+-----------------+------------------+--------------------+
|         148| 1625090.369999921|   925106|1.7566531510982752|       1.653501339299507|  -6.238386951804204|            false|1.6968296053698915|  -3.525607140461393|
|         243|182600.03999999957|   121396|1.5041685063758243|                    NULL|                NULL|             NULL|1.6968296053698915|  11.354180666365082|
|          31|           4318.01|     3192|1.3527600250626568|      0.8802542748103541|  -53.67832497650766|            false|1.6968296053698915|   20.27720280329686